# Food security indices against NDVI, by commune

Pairs the four commonly used household food security indices, aggregated to
commune-month in `data/food_security_monthly.csv`, with the commune NDVI series
in `data/commune_monthly.csv`.

Produces the per-commune panel figures and the correlation table reported in the
supplementary material. All paths are relative to this notebook.

Indices: Food Consumption Score (FCS), Household Dietary Diversity Score
(HDDS, 24-hour recall), Household Hunger Scale (HHS) and reduced Coping
Strategies Index (rCSI).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import seaborn as sns; sns.set()

DATA = "../../data"
INDICES = ["FCS", "HDDS_24hr", "HHS", "rCSI1"]

In [ ]:
# ---------------------------------------------------------------------
# Commune-month indices and NDVI, communes 1-6
# ---------------------------------------------------------------------
fsi = pd.read_csv(f"{DATA}/food_security_monthly.csv")
ndvi = pd.read_csv(f"{DATA}/commune_monthly.csv")[["commune", "YearMonth", "ndvi"]]
commune_lookup = pd.read_csv(f"{DATA}/commune_lookup.csv")
commune_name = dict(zip(commune_lookup["commune"], commune_lookup["commune_name"]))

monthly = fsi.merge(ndvi, on=["commune", "YearMonth"], how="inner")
monthly["Date"] = pd.to_datetime(monthly["YearMonth"])
monthly = monthly.sort_values(["commune", "Date"])

COMMUNES = sorted(monthly["commune"].unique())
print(monthly.groupby("commune").agg(months=("YearMonth", "size"),
                                     min_households=("n_households", "min"),
                                     max_households=("n_households", "max")))


def commune_series(comm):
    """Indices and NDVI for one commune, complete cases only."""
    return monthly[monthly["commune"] == comm].dropna(subset=["ndvi"]).copy()

In [ ]:
# ---------------------------------------------------------------------
# Pearson correlations, NDVI vs each index
# ---------------------------------------------------------------------
rows = []
for comm in COMMUNES:
    df = commune_series(comm)
    for metric in INDICES:
        pair = df[["ndvi", metric]].replace([np.inf, -np.inf], np.nan).dropna()
        r, p = stats.pearsonr(pair["ndvi"], pair[metric])
        rows.append({"commune": comm, "commune_name": commune_name[comm],
                     "metric": metric, "n_pairs": len(pair),
                     "pearson_r": r, "p_value": p})

corr_long = pd.DataFrame(rows)
corr_wide = corr_long.pivot(index="commune", columns="metric", values="pearson_r")

print(corr_long.to_string(index=False,
                          formatters={"pearson_r": "{:.3f}".format,
                                      "p_value": "{:.3f}".format}))
print()
print(corr_wide.round(3).to_string())

corr_long.to_csv("ndvi_food_security_correlations.csv", index=False)

In [ ]:
# ---------------------------------------------------------------------
# One 2x2 panel figure per commune: each index against NDVI
# ---------------------------------------------------------------------
axis_labels = {
    "rCSI1": "Reduced Coping Strategies Index",
    "HDDS_24hr": "Household Dietary Diversity Score",
    "FCS": "Food Consumption Score",
    "HHS": "Household Hunger Score",
}
panel_order = ["rCSI1", "HDDS_24hr", "FCS", "HHS"]
colors = {"rCSI1": "#d4788a", "HDDS_24hr": "#8aad2e", "FCS": "#2aada8", "HHS": "#8080c8"}

plt.rcParams.update({"axes.facecolor": "white", "figure.facecolor": "white"})

for comm in COMMUNES:
    df = commune_series(comm)

    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    for i, (ax, metric) in enumerate(zip(axes.flat, panel_order)):
        ax.grid(False)

        # index on the left axis
        ax.plot(df["Date"], df[metric], color=colors[metric], linewidth=1.2)
        ax.set_ylabel(axis_labels[metric], color="black", fontsize=9)
        ax.tick_params(axis="y", labelcolor="black", labelsize=8)

        # NDVI on the right axis
        ax2 = ax.twinx()
        ax2.grid(False)
        ax2.plot(df["Date"], df["ndvi"], color="black", linewidth=1.5)
        ax2.set_ylabel("NDVI", color="black", fontsize=9)
        ax2.tick_params(axis="y", labelcolor="black", labelsize=8)

        ax.text(0.03, 0.97, "abcd"[i], transform=ax.transAxes,
                fontsize=11, fontweight="bold", va="top", ha="left")

        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.set_xlabel("Year", fontsize=9)
        ax.tick_params(axis="x", labelsize=8)
        for spine in list(ax.spines.values()) + list(ax2.spines.values()):
            spine.set_visible(True); spine.set_edgecolor("black"); spine.set_linewidth(0.75)

    fig.tight_layout()
    fig.savefig(f"fsi_subplots_{commune_name[comm]}.jpg", format="jpg", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()

print("Finished plotting.")